# Lesson 0A: Clustering Foundations Theory

## Introduction

Every day, data arrives without labels:
- Customer transaction records with no category assigned
- News articles with no topic tags
- Genes with no known function
- Satellite images with no ground truth
- User behavior logs with no purchase intent labeled

How do you find structure in unlabeled data? How do you know if a customer group is real, or just noise?

**This is the problem clustering solves.**

Clustering is unsupervised learning — the art of discovering natural groups in data *without* ground-truth labels. Unlike classification, where someone has already said "this email is spam, this one isn't," clustering asks the data itself: *what natural groupings emerge?*

In this lesson, we will:
1. Understand what unsupervised learning is and how it differs from supervised learning
2. Learn to measure distance and similarity between data points
3. Discover the curse of dimensionality and why high-dimensional space is counterintuitive
4. Visualize real data without labels and see patterns with our own eyes
5. Build intuition for why distance metrics matter in clustering

Then in Lesson 0B:
1. Learn how to evaluate clusters without labels
2. Discover internal metrics (silhouette, Davies-Bouldin)
3. Use external metrics when some labels are available
4. Master the techniques for choosing the right number of clusters

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Supervised vs Unsupervised Learning](#supervised-vs-unsupervised)
4. [What Is Clustering?](#what-is-clustering)
5. [Distance Metrics](#distance-metrics)
6. [Euclidean Distance](#euclidean-distance)
7. [Manhattan Distance](#manhattan-distance)
8. [Cosine Similarity](#cosine-similarity)
9. [Comparing Distance Metrics](#comparing-metrics)
10. [The Curse of Dimensionality](#curse-of-dimensionality)
11. [Visualizing Real Data: The Iris Dataset](#iris-dataset)
12. [Distances in Action](#distances-in-action)
13. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

| Library | Purpose |
|---------|----------|
| NumPy | Numerical computing and matrix operations |
| Pandas | Data loading and manipulation |
| Matplotlib | 2D visualization |
| Scikit-learn | Dataset loading and metrics |
| SciPy | Distance computations |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.distance import euclidean, cityblock, cosine
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from typing import Tuple
from numpy.typing import NDArray

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="supervised-vs-unsupervised"></a>
## Supervised vs Unsupervised Learning

### Supervised Learning: "Learning with a Teacher"

In supervised learning, we have labeled examples:
- **Input**: Email text → **Output**: Spam or not spam?
- **Input**: House features → **Output**: Price
- **Input**: Tumor scan → **Output**: Malignant or benign?

A human (or dataset) has already provided the correct answer. The algorithm learns patterns from labeled examples and can predict labels for new, unseen data.

### Unsupervised Learning: "Learning Without a Teacher"

In unsupervised learning, we have only the data — no labels:
- **Input**: Customer purchase histories → **Output**: What natural customer groups exist?
- **Input**: Gene expression levels → **Output**: Which genes behave similarly?
- **Input**: News articles → **Output**: What topics emerge?

No one has told us the answer. The algorithm must find structure within the data itself.

### Side-by-Side Comparison

| Aspect | Supervised | Unsupervised |
|--------|-----------|---------------|
| **Labels** | Provided | None |
| **Problem Type** | Classification or Regression | Clustering or Dimensionality Reduction |
| **Goal** | Predict labels for new data | Discover hidden structure |
| **Evaluation** | Accuracy, precision, recall | Silhouette score, Davies-Bouldin, visual inspection |
| **Interpretability** | Labels are ground truth | Patterns must be interpreted by humans |
| **Real World** | Email spam, medical diagnosis | Customer segmentation, gene discovery |

### Why Unsupervised Learning Matters

Most data in the world is unlabeled. It's expensive and time-consuming to label data by hand. Unsupervised learning lets us:
1. **Explore** data we've never seen before
2. **Discover** unexpected patterns
3. **Reduce complexity** before labeling (label high-potential clusters only)
4. **Understand structure** in domains where labels are undefined

<a name="what-is-clustering"></a>
## What Is Clustering?

**Clustering** is the task of grouping data points such that:
1. Points within the same group are **similar** to each other
2. Points in different groups are **dissimilar** from each other

### A Concrete Example

Imagine you have customers described by two features: annual spending and visit frequency.

```
Customer A: Spend $5000/year, visit 50 times
Customer B: Spend $4800/year, visit 48 times  ← Similar to A
Customer C: Spend $200/year, visit 2 times
Customer D: Spend $180/year, visit 1 time      ← Similar to C
```

Even without labels, we see two natural groups:
- **Cluster 1**: High-spend, high-frequency (premium customers)
- **Cluster 2**: Low-spend, low-frequency (casual customers)

### Why Does This Matter?

Once you've identified clusters, you can:
- **Market differently** to each segment
- **Allocate resources** to high-value groups
- **Understand anomalies** (points that don't fit any cluster)
- **Build targeted models** for each cluster

<a name="distance-metrics"></a>
## Distance Metrics: How We Measure "Similarity"

The foundation of clustering is a simple question: **How do we measure the distance between two points?**

Different distance metrics lead to different clusters. Choose the wrong metric, and you'll find spurious groupings. Choose the right one, and you'll discover genuine structure.

### Mathematical Notation

For two points $p$ and $q$ in $d$-dimensional space:
- $p = (p_1, p_2, \ldots, p_d)$
- $q = (q_1, q_2, \ldots, q_d)$

We want a function $d(p, q)$ that measures how far apart they are.

<a name="euclidean-distance"></a>
## Euclidean Distance: The Straight-Line Distance

**Euclidean distance** is the most common distance metric. It's what you learned in high school geometry: the straight-line distance between two points.

### Formula

$$d_E(p, q) = \sqrt{\sum_{i=1}^{d} (p_i - q_i)^2}$$

In 2D, this is the Pythagorean theorem:
$$d_E(p, q) = \sqrt{(p_x - q_x)^2 + (p_y - q_y)^2}$$

### Intuition

Imagine two cities: City A at coordinates (0, 0) and City B at (3, 4).
- Horizontal distance: 3 units
- Vertical distance: 4 units
- Straight-line distance (Euclidean): $\sqrt{3^2 + 4^2} = \sqrt{25} = 5$ units

### Properties

- **Interpretation**: Real-world distance (miles, meters, etc.)
- **Sensitivity**: Heavily influenced by outliers (large differences are squared)
- **Use case**: When dimensions are comparable in scale (e.g., pixel coordinates, physical measurements)
- **Computational cost**: $O(d)$ per pair

In [ ]:
# Euclidean distance from scratch
def euclidean_distance(p: NDArray, q: NDArray) -> float:
    """
    Compute Euclidean distance between points p and q.
    
    Args:
        p: Point as array [x1, x2, ..., xd]
        q: Point as array [x1, x2, ..., xd]
    
    Returns:
        Euclidean distance
    """
    return np.sqrt(np.sum((p - q) ** 2))

# Example
p = np.array([0, 0])
q = np.array([3, 4])
dist = euclidean_distance(p, q)

print(f"Point p: {p}")
print(f"Point q: {q}")
print(f"Euclidean distance: {dist}")
print(f"Verification (Pythagorean): sqrt(3² + 4²) = {np.sqrt(3**2 + 4**2)}")

<a name="manhattan-distance"></a>
## Manhattan Distance: The Grid-Path Distance

**Manhattan distance** (also called taxicab or L1 distance) measures distance as you'd travel on a grid of city blocks — only horizontal and vertical movement allowed.

### Formula

$$d_M(p, q) = \sum_{i=1}^{d} |p_i - q_i|$$

### Intuition

Using our city example again:
- City A at (0, 0), City B at (3, 4)
- Manhattan distance: $|3 - 0| + |4 - 0| = 3 + 4 = 7$ blocks

Imagine a taxi driver can't cut diagonally through the city — they must follow the grid. They travel 3 blocks east and 4 blocks north, for a total of 7 blocks.

### Properties

- **Interpretation**: Grid-based distance
- **Sensitivity**: Less sensitive to outliers than Euclidean (no squaring)
- **Use case**: When movement is restricted to axes (e.g., city blocks, chessboard moves)
- **Computational cost**: $O(d)$ per pair
- **Comparison**: Always ≥ Euclidean distance (why? see geometry)

In [ ]:
# Manhattan distance from scratch
def manhattan_distance(p: NDArray, q: NDArray) -> float:
    """
    Compute Manhattan distance between points p and q.
    
    Args:
        p: Point as array [x1, x2, ..., xd]
        q: Point as array [x1, x2, ..., xd]
    
    Returns:
        Manhattan distance
    """
    return np.sum(np.abs(p - q))

# Example
p = np.array([0, 0])
q = np.array([3, 4])
dist_euclidean = euclidean_distance(p, q)
dist_manhattan = manhattan_distance(p, q)

print(f"Point p: {p}")
print(f"Point q: {q}")
print(f"Euclidean distance: {dist_euclidean:.2f}")
print(f"Manhattan distance: {dist_manhattan}")
print(f"\nNotice: Manhattan ({dist_manhattan}) ≥ Euclidean ({dist_euclidean:.2f})")

<a name="cosine-similarity"></a>
## Cosine Similarity: The Angle-Based Measure

**Cosine similarity** measures the angle between two vectors, not their magnitude. It answers: "Do these vectors point in the same direction?"

### Formula

$$\text{cosine similarity}(p, q) = \frac{p \cdot q}{\|p\| \|q\|} = \frac{\sum_{i=1}^{d} p_i q_i}{\sqrt{\sum p_i^2} \sqrt{\sum q_i^2}}$$

This ranges from -1 (pointing opposite directions) to +1 (pointing the same direction).

**Cosine distance** is the complement:
$$d_C(p, q) = 1 - \text{cosine similarity}(p, q)$$

### Intuition

Imagine two documents represented as word vectors:
- Document A emphasizes technology, data, AI heavily
- Document B emphasizes the same topics in the same proportions, but with different word counts

Cosine similarity ignores length and focuses on direction. Both documents point "the same way" in topic space, so cosine similarity is high.

### Properties

- **Interpretation**: Angle between vectors; similarity score [0, 1] after distance conversion
- **Sensitivity**: Ignores magnitude (scaled data irrelevant)
- **Use case**: Text analysis, collaborative filtering, comparing directions not magnitudes
- **Computational cost**: $O(d)$ per pair
- **Key insight**: A point very far away in the same direction has cosine similarity = 1 (distance = 0)

In [ ]:
# Cosine similarity from scratch
def cosine_similarity(p: NDArray, q: NDArray) -> float:
    """
    Compute cosine similarity between points p and q.
    
    Returns a value in [-1, 1]:
      1 = same direction
      0 = perpendicular
     -1 = opposite directions
    """
    dot_product = np.dot(p, q)
    magnitude_p = np.linalg.norm(p)
    magnitude_q = np.linalg.norm(q)
    
    if magnitude_p == 0 or magnitude_q == 0:
        return 0.0
    
    return dot_product / (magnitude_p * magnitude_q)

def cosine_distance(p: NDArray, q: NDArray) -> float:
    """Cosine distance: 1 - cosine similarity"""
    return 1 - cosine_similarity(p, q)

# Example 1: Same direction (high similarity)
p = np.array([1, 0, 0])
q = np.array([2, 0, 0])  # Same direction, different magnitude

sim = cosine_similarity(p, q)
dist = cosine_distance(p, q)

print("Example 1: Same direction, different magnitude")
print(f"  p = {p}")
print(f"  q = {q}")
print(f"  Cosine similarity: {sim:.3f}")
print(f"  Cosine distance: {dist:.3f}")

# Example 2: Different directions (low similarity)
p = np.array([1, 0, 0])
q = np.array([0, 1, 0])  # Perpendicular

sim = cosine_similarity(p, q)
dist = cosine_distance(p, q)

print("\nExample 2: Perpendicular directions")
print(f"  p = {p}")
print(f"  q = {q}")
print(f"  Cosine similarity: {sim:.3f}")
print(f"  Cosine distance: {dist:.3f}")

<a name="comparing-metrics"></a>
## Comparing Distance Metrics

### A Visual Comparison

Let's compute all three distances for various pairs of points and see how they differ.

In [ ]:
# Generate some sample points
points = [
    np.array([0, 0]),
    np.array([3, 4]),
    np.array([1, 1]),
    np.array([5, 0]),
]

labels = ['Origin (0,0)', 'P1 (3,4)', 'P2 (1,1)', 'P3 (5,0)']

# Compute distance matrix for each metric
n = len(points)
euclidean_matrix = np.zeros((n, n))
manhattan_matrix = np.zeros((n, n))
cosine_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        euclidean_matrix[i, j] = euclidean_distance(points[i], points[j])
        manhattan_matrix[i, j] = manhattan_distance(points[i], points[j])
        cosine_matrix[i, j] = cosine_distance(points[i], points[j])

print("Euclidean Distance Matrix:")
print(pd.DataFrame(euclidean_matrix, index=labels, columns=labels).round(3))

print("\nManhattan Distance Matrix:")
print(pd.DataFrame(manhattan_matrix, index=labels, columns=labels).round(3))

print("\nCosine Distance Matrix:")
print(pd.DataFrame(cosine_matrix, index=labels, columns=labels).round(3))

In [ ]:
# Visualize the points and distances
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Euclidean visualization
ax = axes[0]
for i, p in enumerate(points):
    ax.scatter(p[0], p[1], s=200, alpha=0.6)
    ax.annotate(f'P{i}', (p[0], p[1]), fontsize=12, ha='center')

# Draw Euclidean circles (points equidistant from origin)
circle = plt.Circle((0, 0), 1, color='red', fill=False, linestyle='--', alpha=0.3)
ax.add_patch(circle)
circle = plt.Circle((0, 0), 3, color='red', fill=False, linestyle='--', alpha=0.3)
ax.add_patch(circle)
ax.set_xlim(-1, 6)
ax.set_ylim(-1, 6)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Euclidean: Circles (equal distance)', fontsize=12, fontweight='bold')
ax.set_xlabel('X')
ax.set_ylabel('Y')

# Manhattan visualization
ax = axes[1]
for i, p in enumerate(points):
    ax.scatter(p[0], p[1], s=200, alpha=0.6)
    ax.annotate(f'P{i}', (p[0], p[1]), fontsize=12, ha='center')

# Draw Manhattan diamonds (points equidistant from origin)
for r in [1, 3]:
    diamond = plt.Polygon([(r, 0), (0, r), (-r, 0), (0, -r)], 
                          fill=False, edgecolor='blue', linestyle='--', alpha=0.3)
    ax.add_patch(diamond)
ax.set_xlim(-1, 6)
ax.set_ylim(-1, 6)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Manhattan: Diamonds (equal distance)', fontsize=12, fontweight='bold')
ax.set_xlabel('X')
ax.set_ylabel('Y')

# Cosine visualization
ax = axes[2]
for i, p in enumerate(points):
    ax.scatter(p[0], p[1], s=200, alpha=0.6)
    ax.annotate(f'P{i}', (p[0], p[1]), fontsize=12, ha='center')
    # Draw line from origin
    if np.linalg.norm(p) > 0:
        ax.plot([0, p[0]], [0, p[1]], 'k-', alpha=0.2)

# Draw angle sectors
for angle in [0, 45, 90, 135]:
    rad = np.radians(angle)
    x = 5 * np.cos(rad)
    y = 5 * np.sin(rad)
    ax.plot([0, x], [0, y], 'g--', alpha=0.2)

ax.set_xlim(-1, 6)
ax.set_ylim(-1, 6)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Cosine: Same direction (equal distance)', fontsize=12, fontweight='bold')
ax.set_xlabel('X')
ax.set_ylabel('Y')

plt.tight_layout()
plt.show()

### Summary: When to Use Each Metric

| Metric | Best For | Example |
|--------|----------|----------|
| **Euclidean** | Physical distance, numerical features at similar scales | 2D point clustering, image pixels |
| **Manhattan** | Grid-based distances, outlier resistance | City block clustering, sparse features |
| **Cosine** | Text, direction matters not magnitude | Document clustering, recommendation systems |

**Golden rule**: Always scale/normalize your data before clustering, especially if features have different units (e.g., age in years vs. income in dollars).

<a name="curse-of-dimensionality"></a>
## The Curse of Dimensionality

Here's a counterintuitive fact: **In high-dimensional space, the notion of "distance" breaks down.**

### The Problem

Imagine you have 1000 data points in 1000-dimensional space (not uncommon in machine learning — think: 1000 documents, each described by 1000 word frequencies).

**Key finding**: All pairwise distances become nearly equal.

This means:
- The nearest neighbor is almost as far as the farthest neighbor
- Clustering becomes nearly meaningless (no natural separation)
- Random points appear to form clusters

### Why?

Let's think about it geometrically:
- In 1D: A ball of radius $r$ contains a line segment
- In 2D: A ball of radius $r$ contains a circle
- In 3D: A ball of radius $r$ contains a sphere
- In 1000D: A ball of radius $r$ contains... almost nothing!

**The volume grows exponentially in high dimensions.** A ball of radius $r$ can fit points that are nearly distance $r$ away in all 1000 dimensions simultaneously.

### Demonstration

In [ ]:
# Demonstrate the curse of dimensionality
dimensions = range(1, 51)
n_samples = 1000

min_distances = []
max_distances = []
mean_distances = []
ratios = []  # max / min

for d in dimensions:
    # Generate random points in d-dimensional space
    X = np.random.rand(n_samples, d)
    
    # Compute all pairwise distances from first point
    distances = np.linalg.norm(X - X[0], axis=1)
    
    min_dist = np.min(distances[1:])  # Exclude distance to itself
    max_dist = np.max(distances[1:])
    mean_dist = np.mean(distances[1:])
    
    min_distances.append(min_dist)
    max_distances.append(max_dist)
    mean_distances.append(mean_dist)
    ratios.append(max_dist / min_dist if min_dist > 0 else 0)

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Min, mean, max distances
ax = axes[0]
ax.plot(dimensions, min_distances, 'g-', linewidth=2, label='Min distance')
ax.plot(dimensions, mean_distances, 'b-', linewidth=2, label='Mean distance')
ax.plot(dimensions, max_distances, 'r-', linewidth=2, label='Max distance')
ax.fill_between(dimensions, min_distances, max_distances, alpha=0.2)
ax.set_xlabel('Number of Dimensions', fontsize=12)
ax.set_ylabel('Distance', fontsize=12)
ax.set_title('Curse of Dimensionality: Distance Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Plot 2: Ratio of max/min
ax = axes[1]
ax.plot(dimensions, ratios, 'purple', linewidth=2)
ax.set_xlabel('Number of Dimensions', fontsize=12)
ax.set_ylabel('Max Distance / Min Distance', fontsize=12)
ax.set_title('Curse of Dimensionality: Ratio', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(y=1, color='k', linestyle='--', alpha=0.3, label='Perfect clustering')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"At d=1:  max/min = {ratios[0]:.2f} (nearby neighbors are clearly closer)")
print(f"At d=10: max/min = {ratios[9]:.2f}")
print(f"At d=50: max/min = {ratios[49]:.2f} (nearby neighbors barely closer than far ones!)")

### Implications for Clustering

1. **Distance becomes meaningless**: If all distances are nearly equal, we can't distinguish "near" from "far"
2. **Clustering becomes random**: Any partition looks about as good as any other
3. **Solution**: Reduce dimensionality first (we'll cover this in later lessons on PCA, t-SNE, UMAP)

**Takeaway**: Always inspect your data's dimensionality. If $d$ is much larger than $\log(n)$ (where $n$ is the number of samples), you likely need dimensionality reduction before clustering.

<a name="iris-dataset"></a>
## Visualizing Real Data: The Iris Dataset

Now let's move from theory to reality. The **Iris dataset** is a classic in machine learning — 150 flowers, each measured on 4 features, and labeled with one of 3 iris species.

For this lesson, we'll **ignore the labels** (we'll use them only to validate our intuition in the next lesson). Our goal: can we visually see natural clusters?

### The Dataset

- **Samples**: 150 iris flowers
- **Features**: Sepal length, sepal width, petal length, petal width (in cm)
- **Classes**: Setosa, Versicolor, Virginica (we'll pretend we don't know these)

### The Challenge

We have 4 dimensions, but humans can't visualize 4D directly. We'll project onto 2D and 3D to see what's there.

In [ ]:
# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target
target_names = iris.target_names

# Create a DataFrame for easier inspection
df = pd.DataFrame(X, columns=iris.feature_names)
df['Species'] = y

print("First few samples:")
print(df.head(10))

print("\nDataset shape:", X.shape)
print("Features:", iris.feature_names)
print("\nSummary statistics:")
print(df.groupby('Species')[iris.feature_names].mean().round(2))

### Visualizing in 2D

In [ ]:
# Pairwise 2D projections
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

feature_pairs = [
    (0, 1),  # Sepal length vs sepal width
    (0, 2),  # Sepal length vs petal length
    (0, 3),  # Sepal length vs petal width
    (1, 2),  # Sepal width vs petal length
    (1, 3),  # Sepal width vs petal width
    (2, 3),  # Petal length vs petal width
]

colors = ['red', 'blue', 'green']

for idx, (i, j) in enumerate(feature_pairs):
    ax = axes[idx]
    
    # Plot each species with a different color (for reference, but pretend we don't know this)
    for species_idx in range(3):
        mask = y == species_idx
        ax.scatter(X[mask, i], X[mask, j], 
                  c=colors[species_idx], label=target_names[species_idx],
                  alpha=0.6, s=60)
    
    ax.set_xlabel(iris.feature_names[i], fontsize=11)
    ax.set_ylabel(iris.feature_names[j], fontsize=11)
    ax.set_title(f'{iris.feature_names[i]} vs {iris.feature_names[j]}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend()

plt.tight_layout()
plt.show()

print("\n💡 Key observation:")
print("Petal measurements (columns 2 and 3) separate the species clearly.")
print("Sepal measurements alone are less separable.")
print("This tells us: feature selection matters for clustering!")

### Visualizing in 3D

In [ ]:
# 3D visualization: Petal length, petal width, sepal length
fig = plt.figure(figsize=(12, 5))

# 3D plot
ax1 = fig.add_subplot(121, projection='3d')

for species_idx in range(3):
    mask = y == species_idx
    ax1.scatter(X[mask, 2], X[mask, 3], X[mask, 0],
               c=colors[species_idx], label=target_names[species_idx],
               alpha=0.6, s=50)

ax1.set_xlabel('Petal length (cm)', fontsize=11)
ax1.set_ylabel('Petal width (cm)', fontsize=11)
ax1.set_zlabel('Sepal length (cm)', fontsize=11)
ax1.set_title('Iris in 3D Space', fontsize=12, fontweight='bold')
ax1.legend()

# 2D projection (for comparison)
ax2 = fig.add_subplot(122)
for species_idx in range(3):
    mask = y == species_idx
    ax2.scatter(X[mask, 2], X[mask, 3],
               c=colors[species_idx], label=target_names[species_idx],
               alpha=0.6, s=60)

ax2.set_xlabel('Petal length (cm)', fontsize=11)
ax2.set_ylabel('Petal width (cm)', fontsize=11)
ax2.set_title('Iris in 2D (Petal Dimensions)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print("\n💡 Key observation:")
print("In 2D (petal dimensions), the three species form nearly distinct clusters.")
print("Adding more dimensions (3D) provides more separation.")
print("This suggests clustering might work well on petal features alone.")

<a name="distances-in-action"></a>
## Distances in Action: The Iris Dataset

In [ ]:
# Standardize features (important for distance-based methods!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Pick a reference flower (first sample) from each species
ref_setosa = X_scaled[0]  # Species 0 (Setosa)
ref_versicolor = X_scaled[50]  # Species 1 (Versicolor)
ref_virginica = X_scaled[100]  # Species 2 (Virginica)

# Compute distances from each reference point to all samples
dist_to_setosa_euclidean = [euclidean_distance(x, ref_setosa) for x in X_scaled]
dist_to_versicolor_euclidean = [euclidean_distance(x, ref_versicolor) for x in X_scaled]
dist_to_virginica_euclidean = [euclidean_distance(x, ref_virginica) for x in X_scaled]

# Compute cosine distances
dist_to_setosa_cosine = [cosine_distance(x, ref_setosa) for x in X_scaled]
dist_to_versicolor_cosine = [cosine_distance(x, ref_versicolor) for x in X_scaled]
dist_to_virginica_cosine = [cosine_distance(x, ref_virginica) for x in X_scaled]

# Plot results
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Euclidean distances
for idx, (dist, label) in enumerate([
    (dist_to_setosa_euclidean, 'Reference: Setosa'),
    (dist_to_versicolor_euclidean, 'Reference: Versicolor'),
    (dist_to_virginica_euclidean, 'Reference: Virginica'),
]):
    ax = axes[0, idx]
    for species_idx in range(3):
        mask = y == species_idx
        ax.scatter(range(len(dist)), np.array(dist)[mask],
                  c=colors[species_idx], alpha=0.6, s=50, label=target_names[species_idx])
    ax.set_ylabel('Euclidean Distance', fontsize=11)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    if idx == 0:
        ax.legend()

# Cosine distances
for idx, (dist, label) in enumerate([
    (dist_to_setosa_cosine, 'Reference: Setosa'),
    (dist_to_versicolor_cosine, 'Reference: Versicolor'),
    (dist_to_virginica_cosine, 'Reference: Virginica'),
]):
    ax = axes[1, idx]
    for species_idx in range(3):
        mask = y == species_idx
        ax.scatter(range(len(dist)), np.array(dist)[mask],
                  c=colors[species_idx], alpha=0.6, s=50, label=target_names[species_idx])
    ax.set_ylabel('Cosine Distance', fontsize=11)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    if idx == 0:
        ax.legend()

plt.tight_layout()
plt.show()

print("\n💡 Key observations:")
print("1. Setosa flowers are close to the Setosa reference (low distance)")
print("2. Versicolor and Virginica are farther from Setosa reference")
print("3. Both Euclidean and Cosine show similar clustering patterns here")
print("4. Distance metric choice would matter more with different data distributions")

<a name="conclusion"></a>
## Conclusion

### What We've Learned

1. **Unsupervised learning** discovers structure in unlabeled data, unlike supervised learning which predicts known labels

2. **Clustering** groups similar points together by measuring distance:
   - **Euclidean**: Straight-line distance (most common)
   - **Manhattan**: Grid-based distance (robust to outliers)
   - **Cosine**: Angle-based similarity (for text and normalized data)

3. **The curse of dimensionality** means distance becomes meaningless in high-dimensional space — all points are nearly equidistant

4. **Real data** (Iris) shows natural clustering in lower-dimensional projections — this is why clustering can work

5. **Feature choice matters**: Petal dimensions separate iris species better than sepal dimensions alone

### Practical Takeaways

- Always normalize/standardize your data before clustering
- Choose your distance metric based on what "similarity" means for your domain
- If your data is very high-dimensional, reduce dimensionality first
- Visualize your data in 2D/3D to build intuition before running algorithms
- No algorithm finds clusters if natural ones don't exist — visual inspection first!

### Next Lesson

In Lesson 0B, we'll learn to evaluate clusters without ground-truth labels. We'll discover:
- **Internal metrics**: Silhouette score, Davies-Bouldin index, Calinski-Harabasz score
- **External metrics**: Adjusted Rand Index, Normalized Mutual Information (when we have partial labels)
- **Techniques for choosing K**: Elbow method, gap statistic

This sets the stage for every clustering algorithm in the curriculum — K-Means, DBSCAN, hierarchical clustering, and beyond.